In [ ]:
import os 
import pandas as pd
import matplotlib.pyplot as plt
import cobra
from scipy import stats
import numpy as np
from scipy.stats import hypergeom
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests


In [ ]:
# read dataframe
RESULT_DIR = "../final_result/predicting_individuals/"
out_path = "../final_result/individual_prediction/avg_importances_individuals_final.csv"
feature_importance = pd.read_csv(out_path)

In [ ]:
# top 20 percent reactions
X = 0.20 
cut = feature_importance["AvgImportance"].quantile(1 - X)
top = feature_importance[feature_importance["AvgImportance"] >= cut]
n_top = len(top)

In [ ]:
feature_importance_top = feature_importance.sort_values(by="AvgImportance", ascending=False).head(1356) # top 20 percent

In [ ]:
# =========================
# Reaction annotation from Mouse-GEM
# =========================

# Relative path to Mouse-GEM model
MODEL_PATH = "../external/mouse-GEM/model/Mouse-GEM.xml"

# Load SBML model
model = cobra.io.read_sbml_model(str(MODEL_PATH))

# Build reaction annotation table.
# Keep only manuscript-relevant information:
# reaction ID, reaction name, and subsystem/pathway.
rows = []

for rxn in model.reactions:
    rows.append({
        "reaction_id": rxn.id,
        "reaction_name": rxn.name,
        "subsystem": getattr(rxn, "subsystem", None),
    })

df_gpr = pd.DataFrame(rows)

# =========================
# Match top RF reactions to Mouse-GEM annotations
# =========================

# Keep a reaction_id column without destroying the original feature-importance table.
# This makes later cells safer.
if "reaction_id" not in feature_importance_top.columns:
    feature_importance_top = feature_importance_top.rename(columns={"Feature": "reaction_id"})

feature_importance_top = feature_importance_top[["reaction_id", "AvgImportance"]].dropna()

# Merge top identity-predictive reactions with reaction annotations.
# This replaces the old gene-based 'merged' object, so later cells using merged still work.
merged = feature_importance_top.merge(df_gpr, on="reaction_id", how="inner")

# Explicit set of top reactions for enrichment / prevalence analyses.
top_rxns = set(merged["reaction_id"])

print("Top reactions in feature-importance table:", feature_importance_top.shape[0])
print("Top reactions found in Mouse-GEM:", merged.shape[0])
print("Unique subsystems represented:", merged["subsystem"].nunique())
display(merged.head())

In [ ]:
RXN_MATRIX_PATH = "../summary_stats_combined/FAC_by_MAR_frac_zero_matrix.csv" # common_rxns from FAC_by_MAR_median_matrix.csv - columns
rxn_universe = pd.read_csv(RXN_MATRIX_PATH, nrows=0).columns
common_rxns = pd.Index(rxn_universe).sort_values()
print(f"Reactions from matrix columns: {len(common_rxns)}")
common_rxns = common_rxns.drop('FAC')

# Subsytem enrichement

In [ ]:
# Universe: all reactions with a subsystem annotation
universe = common_rxns.to_frame(name="reaction_id").merge(df_gpr[["reaction_id", "subsystem"]], on="reaction_id", how="left")
universe = universe[universe["subsystem"].notna()]  # Keep only reactions
N = universe.shape[0]

top_rxns = set(merged["reaction_id"])
n = len(top_rxns)

# Count top per subsystem
sub_counts = universe.groupby("subsystem")["reaction_id"].apply(list)

records = []
for subsystem, rxn_ids in sub_counts.items():
    rxn_set = set(rxn_ids)
    K = len(rxn_set)                 # subsystem size in universe
    k = len(top_rxns & rxn_set)      # hits in top set
    if k == 0:
        continue
    # P(X >= k)
    pval = hypergeom.sf(k-1, N, K, n)
    records.append((subsystem, k, K, n, N, pval))

sub_enr = pd.DataFrame(records, columns=["subsystem","k_hits","K_in_universe","n_top","N_universe","pval"])
sub_enr["fdr_bh"] = sub_enr["pval"].rank(method="min") / len(sub_enr) * 0 + np.nan  # placeholder
# Do proper BH:
sub_enr = sub_enr.sort_values("pval").reset_index(drop=True)
m = len(sub_enr)
bh = sub_enr["pval"].to_numpy() * m / np.arange(1, m + 1)
bh = np.minimum.accumulate(bh[::-1])[::-1]   # enforce monotonicity
sub_enr["fdr_bh"] = np.clip(bh, 0, 1)

print(sub_enr.head(50))


In [ ]:
df_mean = pd.read_csv("../summary_stats_combined/FAC_by_MAR_median_matrix.csv", index_col=0)

In [ ]:
# df_mean: first col = FAC, remaining cols = MARxxxxx reactions
df = df_mean.copy()

# --- Use reaction universe from matrix columns ---
RXN_MATRIX_PATH = "../summary_stats_combined/FAC_by_MAR_frac_zero_matrix.csv"  # columns define reaction universe
rxn_universe = pd.read_csv(RXN_MATRIX_PATH, nrows=0).columns
common_rxns = pd.Index(rxn_universe).sort_values()
print(f"Reactions from matrix columns (including FAC if present): {len(common_rxns)}")

# Drop FAC from reaction list
common_rxns = common_rxns.drop("FAC", errors="ignore")

# Keep only reactions that exist in df_mean (preserve common_rxns order)
rxn_cols = [c for c in common_rxns if c in df.columns]

# Optional diagnostics
missing_in_df = [c for c in common_rxns if c not in df.columns]
if missing_in_df:
    print(f"Reactions in matrix but missing in df_mean: {len(missing_in_df)}")

# Operate on boolean reaction matrix only
# True = reaction has nonzero flux in that mouse
is_present = df[rxn_cols].astype(bool)

# Fraction of mice where reaction has nonzero flux
# Since True=1 and False=0, the mean is the flux-active prevalence
frac_present_by_rxn = is_present.mean(axis=0)

# Fraction of mice where reaction has zero flux
frac_absent_by_rxn = (~is_present).mean(axis=0)

# Reactions that are nonzero for ALL mice
always_present_rxns = frac_present_by_rxn[frac_present_by_rxn == 1.0].index.tolist()

# Reactions that are zero for ALL mice
never_present_rxns = frac_present_by_rxn[frac_present_by_rxn == 0.0].index.tolist()

print("\nMost often present / flux-active (top 20):")
print(frac_present_by_rxn.sort_values(ascending=False).head(20))

print("\nLeast often present / flux-active (top 20):")
print(frac_present_by_rxn.sort_values(ascending=True).head(20))

## How many and which reactions are present

In [ ]:
top_20_rxns = [
    'MAR06298',
    'MAR01331',
    'MAR08741',
    'MAR01530',
    'MAR07875',
    'MAR03952',
    'MAR05032',
    'MAR08530',
    'MAR05458',
    'MAR03869',
    'MAR20135',
    'MAR01917',
    'MAR08559',
    'MAR04345',
    'MAR02121',
    'MAR20148',
    'MAR04344',
    'MAR06100',
    'MAR09403',
    'MAR06277'

]
for rxn in top_20_rxns:
    print(f"{rxn}: {frac_present_by_rxn[rxn]:.3f}")

In [ ]:
def subsystem_presence(
    rxn_annot: pd.DataFrame,
    top_rxns,
    universe_rxns,
    reaction_col="reaction_id",
    subsystem_col="subsystem",
):
    '''
        # Summarize how top RF-important reactions are distributed across subsystems.
        Inputs:
        - rxn_annot: reaction annotation table, e.g. df_gpr
                required columns: reaction_id and subsystem
        - top_rxns: reaction IDs selected as top RF-important reactions
        - universe_rxns: all reaction IDs included in the RF feature universe

        Output:
        - one row per subsystem, with:
        k_hits = number of top reactions in that subsystem
        K_in_universe = number of total RF-universe reactions in that subsystem
        fold_enrichment = observed hits / expected hits under random selection
    '''
    # Convert reaction lists to pandas Index objects for easy intersection/filtering
    top_rxns = pd.Index(top_rxns)
    universe_rxns = pd.Index(universe_rxns)

    # Keep only the columns needed for subsystem counting
    m = rxn_annot[[reaction_col, subsystem_col]].copy()

    # Treat missing subsystem annotations as a separate category
    m[subsystem_col] = m[subsystem_col].fillna("Unannotated")

    # Restrict annotation table to the same reaction universe used by the RF model
    # This prevents enrichment/counts from being biased by reactions that were never tested.
    m = m[m[reaction_col].isin(universe_rxns)]

    # K = total number of RF-universe reactions assigned to each subsystem
    K = (
        m.groupby(subsystem_col)[reaction_col]
        .nunique()
        .rename("K_in_universe")
    )

    # k = number of top RF-important reactions assigned to each subsystem
    m_top = m[m[reaction_col].isin(top_rxns)]
    k = (
        m_top.groupby(subsystem_col)[reaction_col]
        .nunique()
        .rename("k_hits")
    )

    # Combine top-hit counts and universe counts
    out = pd.concat([k, K], axis=1).fillna(0)
    out["k_hits"] = out["k_hits"].astype(int)
    out["K_in_universe"] = out["K_in_universe"].astype(int)

    # Number of top reactions that are actually present in the RF universe
    n_top = len(top_rxns.intersection(universe_rxns))

    # Total number of reactions in the RF universe
    N_universe = len(universe_rxns)

    out["n_top"] = n_top
    out["N_universe"] = N_universe

    # Fraction of all top reactions that fall in each subsystem
    # Example: 0.10 means 10% of top reactions are from this subsystem.
    out["frac_of_top_from_subsystem"] = out["k_hits"] / n_top

    # Fraction of each subsystem's reactions that are top reactions
    # Example: 0.25 means 25% of reactions in this subsystem are top RF reactions.
    out["frac_of_subsystem_in_top"] = np.where(
        out["K_in_universe"] > 0,
        out["k_hits"] / out["K_in_universe"],
        np.nan,
    )

    # Expected number of hits if top reactions were randomly distributed
    # according to subsystem size in the RF universe.
    out["expected_hits"] = n_top * (out["K_in_universe"] / N_universe)

    # Fold enrichment:
    # >1 = more top reactions than expected
    #  1 = as expected
    # <1 = fewer top reactions than expected
    out["fold_enrichment"] = np.where(
        out["expected_hits"] > 0,
        out["k_hits"] / out["expected_hits"],
        np.nan,
    )

    # Rank subsystems by number of top hits, then by enrichment strength
    out = out.sort_values(["k_hits", "fold_enrichment"], ascending=False)

    # Move subsystem names from index into a regular column
    out = out.reset_index().rename(columns={subsystem_col: "subsystem"})

    return out


# Run subsystem summary for top RF-important reactions
presence_df = subsystem_presence(df_gpr, top_rxns, common_rxns)

# Display the main interpretation columns
cols = [
    "subsystem",
    "k_hits",
    "K_in_universe",
    "frac_of_top_from_subsystem",
    "frac_of_subsystem_in_top",
    "fold_enrichment",
]

print(presence_df[cols].head(25).to_string(index=False))

In [ ]:
# subsytems found to be enriched in top reactions (from previous analysis)
target_subsystems = [
        "Transport reactions",       
        "Tyrosine metabolism",       
        "Nucleotide metabolism",     
        "Purine metabolism",         
        "Glycolysis / Gluconeogenesis",      
        "Vitamin B6 metabolism",        
        "Carnitine shuttle (peroxisomal)",        
        "Retinol metabolism",    
        "Arginine and proline metabolism",      
        "Pyrimidine metabolism"      
]


In [ ]:
eps = 1e-12  # treat |mean_flux| <= eps as flux-inactive
n_mice = df_mean.shape[0]

# Use all common reactions, not only top RF-important reactions
reaction_set = set(common_rxns)

# Keep only common reactions that are present in the mean/median flux matrix
rxns_in_matrix = [r for r in reaction_set if r in df_mean.columns]

# Map reactions to Mouse-GEM subsystem annotations
rxn_map = (
    df_gpr.loc[
        df_gpr["reaction_id"].isin(rxns_in_matrix),
        ["reaction_id", "reaction_name", "subsystem"]
    ]
    .drop_duplicates(subset=["reaction_id"])
    .copy()
)

# Restrict to the selected enriched/target subsystems
rxn_map = rxn_map[rxn_map["subsystem"].isin(target_subsystems)].copy()

# Reaction IDs to evaluate
rxn_ids = rxn_map["reaction_id"].tolist()

# Flux-activity matrix:
# True  = reaction has nonzero mean/median flux in that mouse
# False = reaction has zero mean/median flux in that mouse
flux_active = df_mean[rxn_ids].abs().gt(eps)

# Count how many mice have nonzero flux for each reaction
rxn_activity = pd.DataFrame({
    "reaction_id": rxn_ids,
    "subsystem": rxn_map["subsystem"].values,
    "reaction_name": rxn_map["reaction_name"].values,
    "n_mice_flux_active": flux_active.sum(axis=0).values,
})

# Fraction of mice with nonzero flux for each reaction
rxn_activity["frac_mice_flux_active"] = (
    rxn_activity["n_mice_flux_active"] / n_mice
)

# Average absolute mean/median flux across mice
rxn_activity["mean_abs_flux"] = df_mean[rxn_ids].abs().mean(axis=0).values

# Sort within each subsystem by how widely flux-active the reaction is
rxn_activity = rxn_activity.sort_values(
    ["subsystem", "n_mice_flux_active"],
    ascending=[True, False]
)

print(rxn_activity.head(20).to_string(index=False))

# Save full table
rxn_activity.to_csv(
    "reaction_flux_activity_in_target_subsystems_common_rxns.csv",
    index=False
)

In [ ]:
# Summarize flux activity across target subsystems
subsys_activity_summary = (
    rxn_activity.groupby("subsystem")
    .agg(
        n_reactions=("reaction_id", "nunique"),
        mean_mice_flux_active=("n_mice_flux_active", "mean"),
        median_mice_flux_active=("n_mice_flux_active", "median"),
        mean_frac_flux_active=("frac_mice_flux_active", "mean"),
        median_frac_flux_active=("frac_mice_flux_active", "median"),
        mean_abs_flux=("mean_abs_flux", "mean"),
    )
    .reset_index()
    .sort_values("mean_frac_flux_active", ascending=False)
)

print(subsys_activity_summary.to_string(index=False))

subsys_activity_summary.to_csv(
    "subsystem_flux_activity_summary_target_subsystems_common_rxns.csv",
    index=False
)

## How many of the reactions are on/off 

In [ ]:
rxn_with_importance = rxn_activity.merge(
    feature_importance[["Feature", "AvgImportance"]],
    left_on="reaction_id",
    right_on="Feature",
    how="left"
)

# ICC and top reactions

In [ ]:
ICC_reactions =  pd.read_csv('../final_result/reaction_ICC.csv')

In [ ]:
# ---- Standardize reaction id column ----
icc_df = ICC_reactions.copy()

fi_df = feature_importance.copy()

# ---- Merge on reaction id ----
m = icc_df.merge(fi_df[["Feature", "AvgImportance"]], left_on="reaction", right_on="Feature", how="inner")

# Clean
m = m.replace([np.inf, -np.inf], np.nan).dropna(subset=["ICC", "AvgImportance"])
m = m[m["AvgImportance"] >= 0]  # just in case

print("Merged rows:", len(m), "out of ICC:", len(icc_df), "and FI:", len(fi_df))

# ---- Correlations ----
# 1) Spearman (rank)
rho, rho_p = stats.spearmanr(m["ICC"], m["AvgImportance"])

# 2) Pearson on log1p importance
m["logImp"] = np.log1p(m["AvgImportance"])
r, r_p = stats.pearsonr(m["ICC"], m["logImp"])

print(f"Spearman rho(ICC, AvgImportance) = {rho:.4f}, p = {rho_p:.3e}")
print(f"Pearson r(ICC, log1p(AvgImportance)) = {r:.4f}, p = {r_p:.3e}")

# Optional: Kendall tau (more conservative than Spearman)
tau, tau_p = stats.kendalltau(m["ICC"], m["AvgImportance"])
print(f"Kendall tau(ICC, AvgImportance) = {tau:.4f}, p = {tau_p:.3e}")

# ---- Overlap / enrichment: top X% ICC vs top X% importance ----
def overlap_enrichment(df, frac=0.05):
    n = len(df)
    k = max(1, int(np.floor(frac * n)))

    top_icc = set(df.nlargest(k, "ICC")["reaction"])
    top_imp = set(df.nlargest(k, "AvgImportance")["Feature"])

    a = len(top_icc & top_imp)     # overlap
    b = len(top_icc - top_imp)
    c = len(top_imp - top_icc)
    d = n - (a + b + c)

    # Fisher exact (odds ratio, p-value)
    odds, p = stats.fisher_exact([[a, b], [c, d]], alternative="greater")
    return {"n": n, "k_each": k, "overlap": a, "odds_ratio": odds, "p_fisher_greater": p}

for frac in [0.01, 0.05, 0.10]:
    out = overlap_enrichment(m, frac=frac)
    print(f"Top {int(frac*100)}% vs Top {int(frac*100)}% overlap:",
          out["overlap"], "/", out["k_each"],
          "| OR=", out["odds_ratio"], "| p=", out["p_fisher_greater"])

# ---- Save merged table for plotting / reporting ----
m_out = "ICC_vs_feature_importance_merged.csv"
m.to_csv(m_out, index=False)
print("Saved:", m_out)

In [ ]:
# Create figure and axis
fig, ax = plt.subplots(figsize=(6.5, 5), dpi=150)

# Scatter plot: ICC vs reaction importance
ax.scatter(
    m["ICC"],          # x-axis: stability (ICC)
    m["logImp"],       # y-axis: importance (log-transformed)
    s=10,              # small markers
    alpha=0.25,        # transparency to reduce overplotting
    edgecolors="none",
    rasterized=True    # improves performance for large datasets
)

# Axis labels
ax.set_xlabel("ICC", fontsize=11)
ax.set_ylabel("log1p(Reaction Importance)", fontsize=11)

# Clean up plot appearance
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Light grid for readability
ax.grid(True, linestyle="-", linewidth=0.5, alpha=0.2)

# Ticks styling
ax.tick_params(direction="out", length=4, width=0.8, labelsize=10)

# ICC is bounded between 0 and 1
ax.set_xlim(-0.01, 1.01)

# Final layout and display
plt.tight_layout()
plt.show()

# Analysis of the reactions from diet

In [ ]:
# =========================
# CONFIG
# =========================
OUT_DIR = "../final_result"
FINAL_RES_DIR = "../final_result"
MODEL_PATH = "../external/mouse-GEM/model/Mouse-GEM.xml"

IMP_PATH = os.path.join(FINAL_RES_DIR, "avg_importances_diet.csv")         # Feature, AvgImportance
CORR_PATH = os.path.join(FINAL_RES_DIR, "diet_spearman_results_all.csv")   # Feature, Spearman_rho, FDR

FDR_ALPHA = 0.05
TOP_IMPORTANCE_FRACTION = 0.20   # top 20% by importance quantile

# =========================
# LOAD
# =========================
df_imp = pd.read_csv(IMP_PATH)
df_corr = pd.read_csv(CORR_PATH)

# make sure columns are numeric
df_imp["AvgImportance"] = pd.to_numeric(df_imp["AvgImportance"], errors="coerce")
df_corr["Spearman_rho"] = pd.to_numeric(df_corr["Spearman_rho"], errors="coerce")
df_corr["FDR"] = pd.to_numeric(df_corr["FDR"], errors="coerce")

# =========================
# SELECT FEATURES = TOP 20% BY IMPORTANCE RANK (quantile, by count)
# =========================
X = 0.20  # top 20% of reactions by count

cut = df_imp["AvgImportance"].quantile(1 - X)
selected = df_imp[df_imp["AvgImportance"] >= cut].copy()

selected_features = set(selected["Feature"])
print(f"Selected {len(selected_features)} features (top {int(X*100)}% by importance quantile)")

# =========================
# LOAD MODEL + MAP REACTIONS TO SUBSYSTEMS
# =========================
model = cobra.io.read_sbml_model(MODEL_PATH)

rxn_to_sub = {}
all_model_reactions = []
for r in model.reactions:
    all_model_reactions.append(r.id)
    sub = getattr(r, "subsystem", None)
    if sub is not None and str(sub).strip() != "":
        rxn_to_sub[r.id] = str(sub).strip()

# universe = all features in importance file (better than all model reactions for this use-case)
universe_features = set(df_imp["Feature"].dropna().astype(str))
# keep only those that exist in universe
selected_features = selected_features & universe_features

# map all universe features to subsystems
feature_to_sub = {f: rxn_to_sub.get(f, "UNMAPPED") for f in universe_features}

# subsystem -> set(features)
sub_to_features = {}
for f, sub in feature_to_sub.items():
    sub_to_features.setdefault(sub, set()).add(f)

# =========================
# FISHER ENRICHMENT (selected features vs universe)
# =========================
rows = []
M = len(universe_features)   # total universe
N = len(selected_features)   # selected set size

for sub, feats_in_sub in sub_to_features.items():
    a = len(feats_in_sub & selected_features)          # selected AND in subsystem
    b = len(selected_features - feats_in_sub)          # selected AND not in subsystem
    c = len(feats_in_sub - selected_features)          # not selected AND in subsystem
    d = M - (a + b + c)                                # not selected AND not in subsystem

    # skip empty pathways
    if len(feats_in_sub) == 0:
        continue

    # one-sided enrichment test
    odds_ratio, p_value = fisher_exact([[a, b], [c, d]], alternative="greater")

    rows.append({
        "subsystem": sub,
        "a_selected_in_subsystem": a,
        "subsystem_size": len(feats_in_sub),
        "selected_size": N,
        "universe_size": M,
        "odds_ratio": odds_ratio,
        "p_value": p_value
    })

enrich = pd.DataFrame(rows)

# FDR correction
if len(enrich) > 0:
    enrich["q_value"] = multipletests(enrich["p_value"], method="fdr_bh")[1]
    enrich["significant_fdr_0.05"] = enrich["q_value"] < 0.05
else:
    enrich["q_value"] = []
    enrich["significant_fdr_0.05"] = []

# sort and save Fisher results
enrich = enrich.sort_values(["q_value", "p_value"])
fisher_out = os.path.join(OUT_DIR, "subsystem_fisher_enrichment_top20pct_total_importance.csv")
enrich.to_csv(fisher_out, index=False)
print(f"Saved Fisher enrichment: {fisher_out}")

# =========================
# MERGE IMPORTANCE + SPEARMAN + SUBSYSTEM
# =========================
df = pd.merge(df_imp, df_corr, on="Feature", how="inner")
df["subsystem"] = df["Feature"].map(rxn_to_sub).fillna("UNMAPPED")

# keep only Fisher-significant subsystems (optional, simple)
sig_subsystems = set(enrich.loc[enrich["significant_fdr_0.05"] == True, "subsystem"])
df = df[df["subsystem"].isin(sig_subsystems)].copy()

print(f"Remaining features after Fisher-significant subsystem filter: {len(df)}")

# =========================
# SIMPLE DIRECTIONAL SCORE (importance * sign of Spearman), zero if not significant
# =========================
df["Directional_Impact"] = df["AvgImportance"] * np.sign(df["Spearman_rho"])
df.loc[df["FDR"] >= FDR_ALPHA, "Directional_Impact"] = 0

# =========================
# PATHWAY SUMMARY (importance + Spearman direction)
# =========================
pathway_summary = df.groupby("subsystem").agg(
    n_reactions=("Feature", "count"),
    mean_importance=("AvgImportance", "mean"),
    total_importance=("AvgImportance", "sum"),
    net_direction_score=("Directional_Impact", "sum"),
    n_upregulated=("Spearman_rho", lambda x: ((x > 0) & (df.loc[x.index, "FDR"] < FDR_ALPHA)).sum()),
    n_downregulated=("Spearman_rho", lambda x: ((x < 0) & (df.loc[x.index, "FDR"] < FDR_ALPHA)).sum())
)

def classify_pathway(row):
    if row["n_reactions"] < 2:
        return "Too Small"
    if row["total_importance"] == 0:
        return "Not Predictive"
    if row["net_direction_score"] > 0.2 * row["total_importance"]:
        return "Positive diet association"
    if row["net_direction_score"] < -0.2 * row["total_importance"]:
        return "Negative diet association"
    return "Mixed"

pathway_summary["Interpretation"] = pathway_summary.apply(classify_pathway, axis=1)

# add Fisher stats for convenience
pathway_summary = pathway_summary.reset_index().merge(
    enrich[["subsystem", "odds_ratio", "p_value", "q_value", "significant_fdr_0.05"]],
    on="subsystem",
    how="left"
).sort_values("mean_importance", ascending=False)

pathway_out = os.path.join(OUT_DIR, "pathway_directional_summary_top20pct_total_importance.csv")
pathway_summary.to_csv(pathway_out, index=False)
print(f"Saved pathway summary: {pathway_out}")

# =========================
# TOP REACTIONS (by importance) + Spearman interpretation
# =========================
top20 = df.sort_values("AvgImportance", ascending=False).head(20).copy()

def interpret_reaction(row):
    if row["FDR"] >= FDR_ALPHA:
        return "Predictive (Non-linear/Complex)"
    elif row["Spearman_rho"] > 0:
        return "Positive high fat diet association (negative low fat association)"
    else:
        return "Negative high fat diet association (positive low fat association)"

top20["Interpretation"] = top20.apply(interpret_reaction, axis=1)

top20_out = os.path.join(OUT_DIR, "top20_reactions_directional_summary_top20pct_total_importance.csv")
top20[["Feature", "subsystem", "AvgImportance", "Spearman_rho", "FDR", "Interpretation"]].to_csv(top20_out, index=False)
print(f"Saved top reactions summary: {top20_out}")

# =========================
# ALL REACTIONS INSIDE FISHER-SIGNIFICANT SUBSYSTEMS + DIRECTION
# =========================
# (df is already filtered to Fisher-significant subsystems above)

def reaction_direction(row):
    if row["FDR"] >= FDR_ALPHA:
        return "Not significant Spearman"
    elif row["Spearman_rho"] > 0:
        return "Positive high fat diet association (negative low fat association)"
    elif row["Spearman_rho"] < 0:
        return "Negative high fat diet association (positive low fat association)"
    else:
        return "No monotonic direction"

# Add direction column
df["Reaction_Direction"] = df.apply(reaction_direction, axis=1)

# Keep only Fisher-significant subsystems explicitly (already filtered, but this keeps it clear)
sig_subsystems_df = df.copy()

# Add Fisher q/p/odds to each reaction row
sig_subsystems_df = sig_subsystems_df.merge(
    enrich[["subsystem", "odds_ratio", "p_value", "q_value", "significant_fdr_0.05"]],
    on="subsystem",
    how="left"
)

# Sort nicely: subsystem first, then most important reactions
sig_subsystems_df = sig_subsystems_df.sort_values(
    ["q_value", "subsystem", "AvgImportance"],
    ascending=[True, True, False]
)

# Save full reaction-level breakdown for all significant enriched subsystems
sig_rxn_out = os.path.join(OUT_DIR, "all_reactions_in_fisher_significant_subsystems_with_direction.csv")
sig_subsystems_df[[
    "subsystem",
    "Feature",
    "AvgImportance",
    "Spearman_rho",
    "FDR",
    "Reaction_Direction",
    "odds_ratio",
    "q_value"
]].to_csv(sig_rxn_out, index=False)

print(f"Saved reactions in Fisher-significant subsystems: {sig_rxn_out}")

# =========================
# PRINT QUICK RESULTS
# =========================
print("\nTop 10 Fisher-enriched subsystems:")
print(enrich.head(20)[["subsystem", "a_selected_in_subsystem", "subsystem_size", "odds_ratio", "q_value"]])

print("\nTop 10 pathways by mean importance (within Fisher-significant subsystems):")
print(pathway_summary.head(20)[["subsystem", "n_reactions", "mean_importance", "Interpretation", "n_upregulated", "n_downregulated", "q_value"]])

print("\nTop 10 reactions by importance:")
print(top20.head(20)[["Feature", "subsystem", "AvgImportance", "Spearman_rho", "FDR", "Interpretation"]])

## Analysis of top 20 diet-predictive reactions 

In [ ]:
# Top 20 diet-predictive reactions 
top_20_diet = [
    'MAR04109',
    'MAR03793',
    'MAR03792',
    'MAR07809',
    'MAR04437',
    'MAR04656',
    'MAR07639',
    'MAR06235',
    'MAR08981',
    'MAR09844',
    'MAR09884',
    'MAR06238',
    'MAR06779',
    'MAR00181',
    'MAR04332',
    'MAR06240',
    'MAR06237',
    'MAR10408',
    'MAR08980',
    'MAR00229'
]

In [ ]:
spearman_result = pd.read_csv(r"../final_result/diet_spearman_results_all.csv")

In [ ]:
for rxn in top_20_diet:
    print(f"Analyzing reaction: {rxn}")
    subset = spearman_result[spearman_result['Feature'] == rxn]
    print(subset[['Feature', 'Spearman_rho', 'FDR']])

# Example usage for reading/analysing the produced dataframes

In [ ]:
df_gpr = pd.read_csv("../summary_stats/reaction_gene_rules.csv")
df_gpr[df_gpr["subsystem"] == "Beta oxidation of unsaturated fatty acids (n-7) (peroxisomal)"] 

In [ ]:
significant_pathways_diet = pd.read_csv("../final_result/pathway_directional_summary_top20pct_total_importance.csv")
significant_pathways_diet
print(significant_pathways_diet.subsystem.unique())

In [ ]:
significant_pathways_diet[["subsystem", "n_reactions", "q_value", "odds_ratio", "Interpretation", "n_downregulated", "n_upregulated"]].head(20)

In [ ]:
significant_pathways_diet_reactions = pd.read_csv("../final_result/all_reactions_in_fisher_significant_subsystems_with_direction.csv")
significant_pathways_diet_reactions[significant_pathways_diet_reactions["subsystem"] == "Tyrosine metabolism"]
significant_pathways_diet_reactions[significant_pathways_diet_reactions["subsystem"] == "Beta oxidation of unsaturated fatty acids (n-7) (peroxisomal)"]